# New Way — The Tunnel: Path Simulator

## Rules
- Board width = **2 columns** (col 0 = left, col 1 = right)
- Start: `(col=0, row=0)` — bottom-left
- Available moves: **U** (up), **L** (left), **R** (right) — Down is forbidden
  - From col 0: `{U, R}` only
  - From col 1: `{U, L}` only
- **Square-free constraint**: the full move sequence must never contain any repeated consecutive block  
  e.g. `UU`, `RLRL`, `URUR`, `URULURul`, … are all forbidden
- **Score = height** = number of U moves taken (current row when stuck)

## Key finding
Paths are **unbounded**: the search tree never terminates naturally because square-free words  
over a 3-symbol alphabet can be infinitely long (Thue, 1906).  
The game therefore behaves as designed — a player can keep going indefinitely in principle,  
and only gets stuck due to a combination of the board's 2-column constraint and the  
square-free rule eliminating certain local choices.

## What this notebook does
1. **Depth-limited DFS** — finds all *naturally stuck* paths (can't extend) up to a chosen depth, plus counts how many live paths still exist at that depth.
2. **Greedy height search** — heuristic that always prefers U to find very tall paths quickly.
3. **Random tall-path search** — random walk with backtracking to find diverse tall paths.
4. Reports statistics and the longest sequences found.

In [1]:
import random
from collections import Counter, defaultdict

BOARD_WIDTH = 2
DELTA = {'U': (0, 1), 'L': (-1, 0), 'R': (1, 0)}


def has_new_square(s: str) -> bool:
    """
    Check only for squares *ending* at the last character of s.
    Assumes s[:-1] is already square-free (checked on the previous move).
    """
    n = len(s)
    for L in range(1, n // 2 + 1):
        if s[n - 2*L : n - L] == s[n - L : n]:
            return True
    return False


def valid_moves(col: int, seq: str) -> list[str]:
    """Return all moves that are both in-bounds and keep the sequence square-free."""
    moves = []
    for d in ('U', 'L', 'R'):
        dc, _ = DELTA[d]
        nc = col + dc
        if nc < 0 or nc >= BOARD_WIDTH:
            continue
        if not has_new_square(seq + d):
            moves.append(d)
    return moves


def col_from_seq(seq: str) -> int:
    """Recompute the column from a sequence (for verification)."""
    col = 0
    for d in seq:
        col += DELTA[d][0]
    return col

## 1 — Depth-limited DFS: naturally stuck paths

In [2]:
def depth_limited_dfs(depth_limit: int = 150):
    """
    DFS up to `depth_limit` moves.

    Returns
    -------
    stuck   : list of (height, seq) — paths that couldn't extend (naturally stuck)
    live    : int — paths still alive at depth_limit (could potentially continue)
    nodes   : int — total DFS nodes visited
    """
    stuck = []
    live  = 0
    nodes = 0
    stack = [(0, 0, '')]  # (col, row, seq)

    while stack:
        col, row, seq = stack.pop()
        nodes += 1

        if len(seq) >= depth_limit:
            live += 1
            continue

        moves = valid_moves(col, seq)
        if not moves:
            stuck.append((row, seq))
        else:
            for d in moves:
                dc, dr = DELTA[d]
                stack.append((col + dc, row + dr, seq + d))

    stuck.sort(key=lambda x: (x[0], len(x[1])), reverse=True)
    return stuck, live, nodes


DEPTH_LIMIT = 150   # ← adjust; 150 runs in ~2 s, 300 in ~30 s

print(f'Running DFS with depth_limit={DEPTH_LIMIT} ...')
stuck, live_count, nodes = depth_limited_dfs(DEPTH_LIMIT)
print(f'Done.  Nodes visited : {nodes:,}')
print(f'       Stuck paths   : {len(stuck):,}  (naturally cannot extend)')
print(f'       Live paths    : {live_count:,}  (hit the depth limit — could continue)')

Running DFS with depth_limit=150 ...
Done.  Nodes visited : 53,862
       Stuck paths   : 9,097  (naturally cannot extend)
       Live paths    : 878  (hit the depth limit — could continue)


In [3]:
# ── Statistics on naturally stuck paths ──────────────────────────────────────
if stuck:
    heights  = [h        for h, _   in stuck]
    lengths  = [len(seq) for _, seq in stuck]

    max_h  = max(heights)
    max_l  = max(lengths)
    mean_h = sum(heights) / len(heights)
    mean_l = sum(lengths) / len(lengths)

    print('=' * 60)
    print('STUCK-PATH STATISTICS')
    print('=' * 60)
    print(f'  Count             : {len(stuck):,}')
    print(f'  Max height        : {max_h}')
    print(f'  Max seq length    : {max_l}')
    print(f'  Mean height       : {mean_h:.1f}')
    print(f'  Mean seq length   : {mean_l:.1f}')
    print()

    dist_h = Counter(heights)
    print('HEIGHT DISTRIBUTION  (score = number of U moves)')
    print('-' * 50)
    for h in sorted(dist_h.keys(), reverse=True):
        bar = '█' * min(dist_h[h] // max(1, len(stuck) // 50), 50)
        print(f'  h={h:3d}: {dist_h[h]:6d}  {bar}')
    print()

    dist_l = Counter(lengths)
    print('SEQUENCE-LENGTH DISTRIBUTION')
    print('-' * 50)
    for l in sorted(dist_l.keys(), reverse=True):
        bar = '█' * min(dist_l[l] // max(1, len(stuck) // 50), 50)
        print(f'  l={l:3d}: {dist_l[l]:6d}  {bar}')

STUCK-PATH STATISTICS
  Count             : 9,097
  Max height        : 52
  Max seq length    : 149
  Mean height       : 35.0
  Mean seq length   : 104.4

HEIGHT DISTRIBUTION  (score = number of U moves)
--------------------------------------------------
  h= 52:      5  
  h= 51:     83  
  h= 50:    242  █
  h= 49:    366  ██
  h= 48:    381  ██
  h= 47:    351  █
  h= 46:    389  ██
  h= 45:    396  ██
  h= 44:    365  ██
  h= 43:    336  █
  h= 42:    308  █
  h= 41:    310  █
  h= 40:    322  █
  h= 39:    302  █
  h= 38:    280  █
  h= 37:    250  █
  h= 36:    223  █
  h= 35:    234  █
  h= 34:    255  █
  h= 33:    257  █
  h= 32:    278  █
  h= 31:    257  █
  h= 30:    235  █
  h= 29:    229  █
  h= 28:    209  █
  h= 27:    191  █
  h= 26:    181  █
  h= 25:    177  
  h= 24:    163  
  h= 23:    140  
  h= 22:    125  
  h= 21:    113  
  h= 20:    118  
  h= 19:    115  
  h= 18:    102  
  h= 17:    103  
  h= 16:    115  
  h= 15:     98  
  h= 14:     90  
  h= 13:   

In [4]:
# ── Show the top stuck paths ──────────────────────────────────────────────────
TOP_N = 20
print(f'TOP {TOP_N} STUCK PATHS  (by height, then sequence length)')
print('=' * 60)
for h, seq in stuck[:TOP_N]:
    eff = h / len(seq) * 100 if seq else 0
    print(f'  h={h:3d}  len={len(seq):4d}  U-eff={eff:.0f}%')
    print(f'  {seq}')
    print()

TOP 20 STUCK PATHS  (by height, then sequence length)
  h= 52  len= 149  U-eff=35%
  RULURULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULRLURLRULURULRLURULURLRULURULRU

  h= 52  len= 149  U-eff=35%
  URLURULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULRLURLRULURULRLURULURLRULURULRU

  h= 52  len= 149  U-eff=35%
  URLURULURLRULURULRLURULURLRULRLURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURULRU

  h= 52  len= 149  U-eff=35%
  URLURULURLRULURULRLURULURLRULRLURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURULUR

  h= 52  len= 148  U-eff=35%
  URULURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURLRULRLURULURLRULURULRLURULURLU

  h= 51  len= 149  U-eff=34%
  RLRULU

## 2 — Greedy height search
Always prefer U (height gain) when valid, otherwise take the first available lateral move.  
Runs many starts to find tall paths quickly.

In [5]:
def greedy_path(prefer_u: bool = True, max_steps: int = 2000) -> tuple[int, str]:
    """
    Follow a single greedy path.
    prefer_u=True  → try U first, then lateral
    prefer_u=False → try lateral first, then U
    Stops when stuck or max_steps reached.
    """
    col, row, seq = 0, 0, ''
    order = ('U', 'L', 'R') if prefer_u else ('L', 'R', 'U')
    for _ in range(max_steps):
        moved = False
        for d in order:
            dc, dr = DELTA[d]
            nc, nr = col + dc, row + dr
            if nc < 0 or nc >= BOARD_WIDTH or nr < 0:
                continue
            if not has_new_square(seq + d):
                col, row, seq = nc, nr, seq + d
                moved = True
                break
        if not moved:
            break
    return row, seq


h_u, s_u = greedy_path(prefer_u=True)
h_l, s_l = greedy_path(prefer_u=False)

print('GREEDY RESULTS')
print('=' * 60)
print(f'Prefer-U:       h={h_u}  len={len(s_u)}  eff={h_u/len(s_u)*100:.0f}%')
print(f'  {s_u[:120]}', '...' if len(s_u) > 120 else '')
print()
print(f'Prefer-lateral: h={h_l}  len={len(s_l)}  eff={h_l/len(s_l)*100:.0f}%')
print(f'  {s_l[:120]}', '...' if len(s_l) > 120 else '')

GREEDY RESULTS
Prefer-U:       h=4  len=7  eff=57%
  URULURU 

Prefer-lateral: h=3  len=15  eff=20%
  RLRULRLURLRULRL 


## 3 — Random tall-path search with backtracking
Shuffle move order randomly; backtrack when stuck; keep the tallest path found.

In [6]:
def random_tall_search(
    n_trials: int = 500,
    max_steps: int = 2000,
    seed: int = 42,
) -> list[tuple[int, str]]:
    """
    Run n_trials random walks, each going up to max_steps moves.
    At each step, shuffle the valid moves randomly and pick the first.
    Returns the top-20 distinct paths by height.
    """
    rng = random.Random(seed)
    best: list[tuple[int, str]] = []

    for _ in range(n_trials):
        col, row, seq = 0, 0, ''
        for _ in range(max_steps):
            moves = valid_moves(col, seq)
            if not moves:
                break
            d = rng.choice(moves)
            dc, dr = DELTA[d]
            col, row, seq = col + dc, row + dr, seq + d
        best.append((row, seq))

    best.sort(key=lambda x: (x[0], len(x[1])), reverse=True)
    # Deduplicate sequences (keep unique top results)
    seen, deduped = set(), []
    for h, seq in best:
        if seq not in seen:
            seen.add(seq)
            deduped.append((h, seq))
    return deduped[:20]


print('Running random search (500 trials) ...')
top_random = random_tall_search(n_trials=500, max_steps=2000, seed=42)
print('Done.')
print()
print('TOP RESULTS FROM RANDOM SEARCH')
print('=' * 60)
for h, seq in top_random[:10]:
    eff = h / len(seq) * 100 if seq else 0
    stuck_marker = '' if valid_moves(col_from_seq(seq), seq) else '  ← STUCK'
    print(f'  h={h:4d}  len={len(seq):5d}  eff={eff:.0f}%{stuck_marker}')
    print(f'  {seq[:120]}', '...' if len(seq) > 120 else '')
    print()

Running random search (500 trials) ...
Done.

TOP RESULTS FROM RANDOM SEARCH
  h=  24  len=   73  eff=33%  ← STUCK
  URLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULRLURLRULURLR 

  h=  21  len=   60  eff=35%  ← STUCK
  URLRULURULRLURULURLRULURULRLURLRULRLURULURLRULRLURLRULURULRU 

  h=  20  len=   63  eff=32%  ← STUCK
  RULRLURLRULURULRLURLRULRLURULURLRULRLURLRULURULRLURLRULRLURULUR 

  h=  19  len=   57  eff=33%  ← STUCK
  URLURULRLURLRULRLURULURLRULRLURLRULURULRLURLRULRLURULURLU 

  h=  17  len=   49  eff=35%  ← STUCK
  URLURULRLURLRULRLURULURLRULRLURLRULURULRLURULURLU 

  h=  16  len=   49  eff=33%  ← STUCK
  RLRULRLURULURLRULRLURLRULURULRLURULURLRULRLURULRU 

  h=  16  len=   47  eff=34%  ← STUCK
  RULURLRULURULRLURLRULRLURULURLRULRLURLRULURULRU 

  h=  16  len=   47  eff=34%  ← STUCK
  RULURULRLURULURLRULRLURLRULURULRLURLRULRLURULRU 

  h=  16  len=   44  eff=36%  ← STUCK
  URLRULURULRLURULURLRULURULRLURLRULRLURULURLU 

  h=  15  len=   48  eff=31%  ← STUCK
  RLURULR

## 4 — Summary

In [7]:
all_found = stuck + top_random
all_found.sort(key=lambda x: (x[0], len(x[1])), reverse=True)

best_h  = all_found[0][0]  if all_found else 0
best_seq = all_found[0][1] if all_found else ''

print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'  Paths are UNBOUNDED — the game can go on indefinitely.')
print(f'  (At depth={DEPTH_LIMIT}, {live_count:,} paths were still alive.)')
print()
print(f'  Within depth={DEPTH_LIMIT}, naturally stuck paths: {len(stuck):,}')
print(f'  Max height in stuck paths : {max(h for h,_ in stuck) if stuck else 0}')
print()
print(f'  Best height from random search (2000 steps/trial): {top_random[0][0] if top_random else 0}')
print()
print(f'  Overall best path found:')
print(f'    height={best_h}  len={len(best_seq)}  U-eff={best_h/len(best_seq)*100:.0f}%' if best_seq else '  (none)')
print(f'    {best_seq[:160]}', '...' if len(best_seq) > 160 else '')

SUMMARY
  Paths are UNBOUNDED — the game can go on indefinitely.
  (At depth=150, 878 paths were still alive.)

  Within depth=150, naturally stuck paths: 9,097
  Max height in stuck paths : 52

  Best height from random search (2000 steps/trial): 24

  Overall best path found:
    height=52  len=149  U-eff=35%
    RULURULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULURULRLURULURLRULRLURLRULURULRLURULURLRULURULRLURLRULRLURULURLRULRLURLRULURULRLURULURLRULURULRU 
